# Sprint 1 - EDA
The four Kaggle datasets used by the travel planner (sources in `docs/dataset_sources.md`).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src import config
from src.data_loader import load_customers, load_flights, load_attractions, load_hotels
pd.set_option('display.max_columns', 50)

## 1. Customer dataset (Holiday Package Purchase Prediction) (preference model)

In [ ]:
cust = load_customers()
print(cust.shape)
cust.head()

In [ ]:
cust.info()

### Missing values

In [ ]:
cust.isna().sum().sort_values(ascending=False).head(12)

### Target: `ProductPitched` (package tier)

In [ ]:
ax = cust[config.TARGET].value_counts().reindex(config.TIER_ORDER).plot.bar(color='#2a78b5')
ax.set_title('Package tier distribution'); plt.show()

### Income and age by tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cust.boxplot('MonthlyIncome', by=config.TARGET, ax=axes[0])
cust.boxplot('Age', by=config.TARGET, ax=axes[1])
plt.suptitle(''); plt.show()

### Designation vs tier
`Designation` maps 1:1 to `ProductPitched`, so it is not used as a feature (leakage).

In [ ]:
pd.crosstab(cust['Designation'], cust[config.TARGET])

### Categorical features

In [ ]:
for c in config.CATEGORICAL_FEATURES:
    print(cust[c].value_counts(), '\n')

## 2. Flight dataset (Flight Price Prediction) (transport agent)

In [ ]:
fl = load_flights()
print(fl.shape)
fl.head()

In [ ]:
fl.groupby(['source_city','destination_city'])['price'].median().unstack().round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fl.groupby('days_left')['price'].mean().plot(ax=axes[0], title='Avg price vs days before departure')
fl.groupby('class')['price'].mean().plot.bar(ax=axes[1], title='Avg price by class', color='#e8913a')
plt.show()

## 3. Attractions dataset (India's Must-See Places) (attraction agent, RAG)

In [ ]:
att = load_attractions()
print(att.shape)
att.head()

In [ ]:
att['city'].value_counts().head(15).plot.barh(title='Cities with most attractions'); plt.show()

In [ ]:
att['significance'].value_counts().plot.bar(title='Attraction significance', color='#2a78b5'); plt.show()

In [ ]:
att[['rating','fee','hours']].describe()

## 4. Hotel dataset (Hotels on Goibibo) (hotel agent, RAG)

In [ ]:
ho = load_hotels()
print(ho.shape, '| price estimated from stars:', bool(ho['price_estimated'].all()))
ho.head()

In [ ]:
ho['city'].value_counts().head(15).plot.barh(title='Cities with most hotels'); plt.show()

In [ ]:
ho['stars'].value_counts().sort_index().plot.bar(title='Star rating distribution', color='#e8913a'); plt.show()

## 5. Destination cities that have both attractions and hotels

In [ ]:
both = sorted(set(att['city'].str.lower()) & set(ho['city'].str.lower()))
print(len(both), 'cities:', both[:40])

## Observations
- 4,888 customers. Basic (37.7%) and Deluxe (35.4%) dominate, King is only 4.7%, so we use macro-F1 and a stratified split.
- Age (226) and MonthlyIncome (233) have the most missing values, filled with the median in the pipeline.
- Gender has a "Fe Male" typo (155 rows) and MaritalStatus has both "Single" and "Unmarried"; both are cleaned in the loader.
- Median income rises with the tier (about Rs 20.7k for Basic to Rs 35k for King).
- Designation is a perfect proxy for the tier (Executive = Basic, Manager = Deluxe, ...), dropped.
- Flights: 300k fares on 30 routes between 6 metros. Economy fares booked 1-3 days ahead average ~Rs 13.8k vs ~Rs 4.9k at 30+ days. 84% of flights have one stop.
- Hotels: 4,000 hotels in 558 cities, 48% unrated, no price column. Latitude/longitude are available and used for city distances.
